# ViT 실험 (An Image is Worth 16x16 Words)

코랩 메뉴 **런타임 > 런타임 유형 변경**에서 GPU(T4)를 고른 뒤 위에서부터 차례로 실행하세요.

- 실험 1: 작은 ViT와 ResNet-20을 CIFAR-10으로 처음부터 학습
- 실험 2: ImageNet-21k로 사전학습된 ViT-B/16 위에 선형 분류기만 학습
- 실험 3: 사전학습된 ViT의 위치 임베딩, attention 시각화

노트북 CPU에서는 8 epoch만 돌렸지만, GPU에서는 아래처럼 30 epoch와 더 큰 ViT로 돌릴 수 있습니다.

In [ ]:
!pip -q install timm
import torch
print(torch.__version__, torch.cuda.is_available())

## 코드 파일 만들기

In [ ]:
%%writefile models.py
"""실험에 쓰는 두 모델: 작은 ViT와 CIFAR용 ResNet-20.

ViT는 논문(Dosovitskiy et al., 2021)의 식 (1)~(4)를 그대로 옮긴 것이다.
  (1) z_0 = [x_class; x_p^1 E; ...; x_p^N E] + E_pos
  (2) z'_l = MSA(LN(z_{l-1})) + z_{l-1}
  (3) z_l  = MLP(LN(z'_l)) + z'_l
  (4) y    = LN(z_L^0)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Vision Transformer
# ----------------------------------------------------------------------------
class PatchEmbedding(nn.Module):
    """이미지를 P x P 패치로 자르고, 각 패치를 D차원 벡터로 선형 변환한다.

    kernel_size = stride = P 인 합성곱은 "패치를 펴서 행렬 E를 곱하는 것"과 같은 연산이다.
    """

    def __init__(self, image_size, patch_size, dim, in_channels=3):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):                      # (B, 3, H, W)
        x = self.proj(x)                       # (B, D, H/P, W/P)
        return x.flatten(2).transpose(1, 2)    # (B, N, D)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5    # 1 / sqrt(d_k)
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                   # 각각 (B, heads, N, d_k)
        attn = (q @ k.transpose(-2, -1)) * self.scale      # QK^T / sqrt(d_k)
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, D)    # head들을 이어 붙임
        return self.out(x)


class EncoderBlock(nn.Module):
    """논문 Figure 1 오른쪽: Norm -> MSA -> (+) -> Norm -> MLP -> (+)"""

    def __init__(self, dim, heads, mlp_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))       # 식 (2)
        x = x + self.mlp(self.norm2(x))        # 식 (3)
        return x


class ViT(nn.Module):
    def __init__(self, image_size=32, patch_size=4, num_classes=10,
                 dim=192, depth=6, heads=3, mlp_dim=384, dropout=0.0,
                 use_pos_embedding=True):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))            # x_class
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, dim))        # E_pos
        self.use_pos_embedding = use_pos_embedding
        self.blocks = nn.ModuleList([EncoderBlock(dim, heads, mlp_dim, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x)                                  # (B, N, D)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)                           # (B, N+1, D)
        if self.use_pos_embedding:
            x = x + self.pos_embed                               # 식 (1)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x[:, 0]))                     # 식 (4): [class] 토큰만 사용


# ----------------------------------------------------------------------------
# ResNet-20 (He et al., 2016의 CIFAR용 ResNet)
# ----------------------------------------------------------------------------
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + (x if self.shortcut is None else self.shortcut(x)))


class ResNet20(nn.Module):
    def __init__(self, num_classes=10, width=16):
        super().__init__()
        self.conv = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(width)
        layers, in_ch = [], width
        for out_ch, stride in [(width, 1), (width * 2, 2), (width * 4, 2)]:
            for i in range(3):
                layers.append(BasicBlock(in_ch, out_ch, stride if i == 0 else 1))
                in_ch = out_ch
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.layers(x)
        return self.fc(x.mean(dim=(2, 3)))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile exp1_scratch.py
"""실험 1: CIFAR-10을 처음부터(사전학습 없이) 학습 - 작은 ViT 대 ResNet-20.

논문의 주장 "데이터가 적으면 ViT는 CNN보다 못하다"를 작은 규모로 확인한다.

사용 예
  python exp1_scratch.py --model vit --epochs 10
  python exp1_scratch.py --model resnet --epochs 10
  python exp1_scratch.py --model vit --no-pos --epochs 10      (위치 임베딩 제거)
"""
import argparse, json, math, os, time
import torch
import torch.nn.functional as F
import torchvision

from models import ViT, ResNet20, count_parameters

MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


def load_cifar10(root, device):
    """CIFAR-10 전체를 텐서로 메모리에 올린다 (데이터 로더보다 빠르다)."""
    def to_tensor(ds):
        x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float().div(255)
        return ((x - MEAN) / STD).to(device), torch.tensor(ds.targets).to(device)
    train = torchvision.datasets.CIFAR10(root, train=True, download=True)
    test = torchvision.datasets.CIFAR10(root, train=False, download=True)
    return to_tensor(train), to_tensor(test)


def augment(x):
    """무작위 좌우 반전 + 4픽셀 패딩 후 무작위 자르기."""
    B = x.shape[0]
    flip = torch.rand(B, device=x.device) < 0.5
    x = torch.where(flip.view(B, 1, 1, 1), x.flip(3), x)
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i, j = torch.randint(0, 9, (2,))
    return x[:, :, i:i + 32, j:j + 32]


@torch.no_grad()
def evaluate(model, x, y, batch_size=1000):
    model.eval()
    correct = 0
    for k in range(0, len(x), batch_size):
        correct += (model(x[k:k + batch_size]).argmax(1) == y[k:k + batch_size]).sum().item()
    return 100.0 * correct / len(x)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", choices=["vit", "resnet"], required=True)
    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--train-size", type=int, default=50000)
    p.add_argument("--dim", type=int, default=192)
    p.add_argument("--depth", type=int, default=6)
    p.add_argument("--heads", type=int, default=3)
    p.add_argument("--patch", type=int, default=4)
    p.add_argument("--no-pos", action="store_true", help="위치 임베딩을 쓰지 않는다")
    p.add_argument("--data", default=os.path.join(os.path.expanduser("~"), ".cache", "vit_exp"))
    p.add_argument("--out", default="results")
    p.add_argument("--name", default=None)
    p.add_argument("--seed", type=int, default=0)
    args = p.parse_args()

    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(args.out, exist_ok=True)
    name = args.name or (args.model + ("_nopos" if args.no_pos else ""))

    (xtr, ytr), (xte, yte) = load_cifar10(args.data, device)
    xtr, ytr = xtr[:args.train_size], ytr[:args.train_size]

    if args.model == "vit":
        model = ViT(patch_size=args.patch, dim=args.dim, depth=args.depth, heads=args.heads,
                    mlp_dim=args.dim * 2, use_pos_embedding=not args.no_pos).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
        warmup_epochs = 1
    else:
        model = ResNet20().to(device)
        opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
        warmup_epochs = 0

    steps_per_epoch = math.ceil(len(xtr) / args.batch_size)
    total, warm = args.epochs * steps_per_epoch, warmup_epochs * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))

    log = {"name": name, "model": args.model, "params": count_parameters(model), "device": device,
           "args": vars(args), "epochs": []}
    print(f"[{name}] 파라미터 {log['params']:,}개, 장치 {device}, 학습 이미지 {len(xtr):,}장", flush=True)

    start = time.time()
    for epoch in range(1, args.epochs + 1):
        model.train()
        perm = torch.randperm(len(xtr), device=device)
        loss_sum, correct = 0.0, 0
        for k in range(0, len(xtr), args.batch_size):
            idx = perm[k:k + args.batch_size]
            x, y = augment(xtr[idx]), ytr[idx]
            out = model(x)
            loss = F.cross_entropy(out, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if args.model == "vit":
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            loss_sum += loss.item() * len(idx)
            correct += (out.argmax(1) == y).sum().item()
        test_acc = evaluate(model, xte, yte)
        rec = {"epoch": epoch, "train_loss": loss_sum / len(xtr), "train_acc": 100.0 * correct / len(xtr),
               "test_acc": test_acc, "elapsed_min": (time.time() - start) / 60}
        log["epochs"].append(rec)
        print(f"[{name}] epoch {epoch:2d}/{args.epochs}  loss {rec['train_loss']:.3f}  "
              f"train {rec['train_acc']:.1f}%  test {test_acc:.2f}%  ({rec['elapsed_min']:.1f}분)", flush=True)
        with open(os.path.join(args.out, f"{name}.json"), "w", encoding="utf-8") as f:
            json.dump(log, f, ensure_ascii=False, indent=1)
    torch.save(model.state_dict(), os.path.join(args.out, f"{name}.pt"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile exp2_pretrained.py
"""실험 2: ImageNet-21k로 사전학습된 ViT-B/16을 CIFAR-10에 옮겨 쓰기 (linear probe).

논문 저자들이 공개한 사전학습 가중치(timm의 vit_base_patch16_224.orig_in21k)를 내려받아
가중치는 그대로 둔 채(frozen) 마지막 [class] 토큰 표현만 뽑고, 그 위에 선형 분류기 하나만 학습한다.
논문 4.3절의 "linear few-shot evaluation"과 같은 방식이다.

사용 예
  python exp2_pretrained.py --train-size 5000 --test-size 2000
"""
import argparse, json, os, time
import torch
import torch.nn.functional as F
import torchvision
import timm

MODEL = "vit_base_patch16_224.orig_in21k"


@torch.no_grad()
def extract_features(model, images, device, batch_size=50, tag=""):
    """images: uint8 (N, 32, 32, 3) numpy 배열 -> (N, 768) 특징."""
    feats, start = [], time.time()
    for k in range(0, len(images), batch_size):
        x = torch.from_numpy(images[k:k + batch_size]).permute(0, 3, 1, 2).float().div(255).to(device)
        x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)   # 32x32 -> 224x224
        x = (x - 0.5) / 0.5                                                    # 논문 가중치의 정규화
        feats.append(model(x).cpu())
        done = k + len(x)
        if (k // batch_size) % 10 == 0 or done == len(images):
            el = time.time() - start
            print(f"  [{tag}] {done}/{len(images)}  {done / el:.1f} img/s  남은 시간 {(len(images) - done) / (done / el) / 60:.1f}분", flush=True)
    return torch.cat(feats)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--train-size", type=int, default=5000)
    p.add_argument("--test-size", type=int, default=2000)
    p.add_argument("--data", default=os.path.join(os.path.expanduser("~"), ".cache", "vit_exp"))
    p.add_argument("--out", default="results")
    args = p.parse_args()

    torch.manual_seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(args.out, exist_ok=True)

    train = torchvision.datasets.CIFAR10(args.data, train=True, download=True)
    test = torchvision.datasets.CIFAR10(args.data, train=False, download=True)
    g = torch.Generator().manual_seed(0)
    tr_idx = torch.randperm(len(train.data), generator=g)[:args.train_size].numpy()
    te_idx = torch.randperm(len(test.data), generator=g)[:args.test_size].numpy()
    ytr = torch.tensor(train.targets)[tr_idx]
    yte = torch.tensor(test.targets)[te_idx]

    model = timm.create_model(MODEL, pretrained=True, num_classes=0).eval().to(device)
    print(f"모델 {MODEL}, 파라미터 {sum(p.numel() for p in model.parameters()):,}개, 장치 {device}", flush=True)

    start = time.time()
    ftr = extract_features(model, train.data[tr_idx], device, tag="train")
    fte = extract_features(model, test.data[te_idx], device, tag="test")
    extract_min = (time.time() - start) / 60

    # 선형 분류기 (768 -> 10) 하나만 학습
    mean, std = ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6
    ftr, fte = (ftr - mean) / std, (fte - mean) / std
    clf = torch.nn.Linear(ftr.shape[1], 10)
    opt = torch.optim.LBFGS(clf.parameters(), lr=1.0, max_iter=200, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(clf(ftr), ytr) + 1e-3 * clf.weight.pow(2).sum()
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        train_acc = 100.0 * (clf(ftr).argmax(1) == ytr).float().mean().item()
        test_acc = 100.0 * (clf(fte).argmax(1) == yte).float().mean().item()

    result = {"name": "vit_b16_in21k_linear_probe", "model": MODEL, "train_size": args.train_size,
              "test_size": args.test_size, "train_acc": train_acc, "test_acc": test_acc,
              "extract_min": extract_min, "device": device}
    print(f"linear probe: 학습 {train_acc:.2f}%  테스트 {test_acc:.2f}%  (특징 추출 {extract_min:.1f}분)", flush=True)
    with open(os.path.join(args.out, "pretrained.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile exp3_visualize.py
"""실험 3: 사전학습된 ViT-B/16 내부 들여다보기 (논문 Figure 6, 7 재현).

  (a) 위치 임베딩 유사도   - 논문 Figure 7 가운데
  (b) 패치 임베딩 필터     - 논문 Figure 7 왼쪽 (주성분 28개)
  (c) attention rollout    - 논문 Figure 6 (부록 D.7의 방법)

사용 예
  python exp3_visualize.py --images sample_images/astronaut.jpg sample_images/dog.jpg
"""
import argparse, os
import numpy as np
import torch
import torch.nn.functional as F
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

MODEL = "vit_base_patch16_224.orig_in21k"


def position_embedding_similarity(model, out):
    pos = model.pos_embed[0, 1:].detach()                 # [class] 제외, (196, 768)
    sim = F.cosine_similarity(pos[:, None, :], pos[None, :, :], dim=-1).reshape(14, 14, 14, 14).numpy()
    fig, axes = plt.subplots(14, 14, figsize=(7.2, 7.2))
    for i in range(14):
        for j in range(14):
            ax = axes[i, j]
            im = ax.imshow(sim[i, j], vmin=-1, vmax=1, cmap="viridis")
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Position embedding similarity (ViT-B/16)", fontsize=14)
    fig.supxlabel("Input patch column"); fig.supylabel("Input patch row")
    fig.subplots_adjust(left=0.07, right=0.88, top=0.93, bottom=0.06, wspace=0.12, hspace=0.12)
    cax = fig.add_axes([0.90, 0.06, 0.02, 0.87])
    fig.colorbar(im, cax=cax, label="Cosine similarity")
    fig.savefig(os.path.join(out, "pos_embed_similarity.png"), dpi=200)
    plt.close(fig)


def patch_embedding_filters(model, out):
    w = model.patch_embed.proj.weight.detach()            # (768, 3, 16, 16)
    flat = w.reshape(w.shape[0], -1)
    flat = flat - flat.mean(0, keepdim=True)
    _, _, v = torch.pca_lowrank(flat, q=28, center=False)
    comps = v.T.reshape(28, 3, 16, 16)
    fig, axes = plt.subplots(4, 7, figsize=(7, 4.3))
    for ax, c in zip(axes.flat, comps):
        c = (c - c.min()) / (c.max() - c.min())
        ax.imshow(c.permute(1, 2, 0).numpy(), interpolation="nearest")
        ax.axis("off")
    fig.suptitle("RGB embedding filters (first 28 principal components, ViT-B/16)", fontsize=11)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.02, wspace=0.08, hspace=0.08)
    fig.savefig(os.path.join(out, "patch_embed_filters.png"), dpi=200)
    plt.close(fig)


@torch.no_grad()
def attention_rollout(model, x):
    """부록 D.7: head 평균 attention에 항등행렬을 더해 층마다 곱해 나간다."""
    inputs = []
    hooks = [b.norm1.register_forward_hook(lambda m, i, o: inputs.append(o)) for b in model.blocks]
    model(x)
    for h in hooks:
        h.remove()
    rollout = None
    for block, z in zip(model.blocks, inputs):
        a = block.attn
        B, N, D = z.shape
        qkv = a.qkv(z).reshape(B, N, 3, a.num_heads, D // a.num_heads).permute(2, 0, 3, 1, 4)
        attn = ((qkv[0] @ qkv[1].transpose(-2, -1)) * (D // a.num_heads) ** -0.5).softmax(-1)
        attn = attn.mean(1)                                # head 평균
        attn = attn + torch.eye(N)                         # residual connection 반영
        attn = attn / attn.sum(-1, keepdim=True)
        rollout = attn if rollout is None else attn @ rollout
    mask = rollout[0, 0, 1:].reshape(14, 14)               # [class] 토큰 -> 각 패치
    # 보기 쉽도록 최솟값 0, 최댓값 1로 맞춘다 (논문 코드는 최댓값으로만 나눈다)
    return ((mask - mask.min()) / (mask.max() - mask.min())).numpy()


def attention_maps(model, paths, out):
    fig, axes = plt.subplots(len(paths), 2, figsize=(5, 2.5 * len(paths)), squeeze=False)
    for row, path in zip(axes, paths):
        img = Image.open(path).convert("RGB")
        s = min(img.size)
        l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((224, 224), Image.BICUBIC)
        arr = np.asarray(img).astype(np.float32) / 255
        x = torch.from_numpy(arr).permute(2, 0, 1)[None]
        mask = attention_rollout(model, (x - 0.5) / 0.5)
        mask = np.asarray(Image.fromarray((mask * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255
        row[0].imshow(arr); row[1].imshow(arr * mask[..., None])
        for ax in row:
            ax.axis("off")
    axes[0, 0].set_title("Input"); axes[0, 1].set_title("Attention")
    fig.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.02, wspace=0.05, hspace=0.08)
    fig.savefig(os.path.join(out, "attention_rollout.png"), dpi=200)
    plt.close(fig)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--images", nargs="*", default=[])
    p.add_argument("--out", default="results")
    args = p.parse_args()
    os.makedirs(args.out, exist_ok=True)
    model = timm.create_model(MODEL, pretrained=True, num_classes=0).eval()
    position_embedding_similarity(model, args.out); print("pos_embed_similarity.png", flush=True)
    patch_embedding_filters(model, args.out); print("patch_embed_filters.png", flush=True)
    if args.images:
        attention_maps(model, args.images, args.out); print("attention_rollout.png", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile plot_results.py
"""results/*.json을 읽어 발표용 그래프를 그린다."""
import json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

for f in ["Malgun Gothic", "NanumGothic", "NanumBarunGothic"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
plt.rcParams["axes.unicode_minus"] = False

R = "results"
load = lambda n: json.load(open(os.path.join(R, n), encoding="utf-8"))
vit, res = load("vit.json"), load("resnet.json")
pre = load("pretrained.json") if os.path.exists(os.path.join(R, "pretrained.json")) else None
C_VIT, C_RES, C_PRE = "#D55E00", "#7F7F7F", "#0072B2"

# (1) 학습 곡선
fig, ax = plt.subplots(figsize=(6.4, 4.2))
curves = [(res, C_RES, "ResNet-20 (처음부터 학습)"), (vit, C_VIT, "작은 ViT (처음부터 학습)")]
if os.path.exists(os.path.join(R, "vit_nopos.json")):
    curves.append((load("vit_nopos.json"), "#F0B48A", "작은 ViT (위치 임베딩 없음)"))
for log, c, label in curves:
    ep = [e["epoch"] for e in log["epochs"]]
    ax.plot(ep, [e["test_acc"] for e in log["epochs"]], "o-", color=c, lw=2, label=label)
if pre:
    ax.axhline(pre["test_acc"], color=C_PRE, lw=2, ls="--", label="사전학습 ViT-B/16 (linear probe)")
ax.set_xlabel("Epoch"); ax.set_ylabel("CIFAR-10 테스트 정확도 (%)")
ax.set_ylim(30, 100); ax.grid(alpha=0.3); ax.legend(loc="lower right", frameon=False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(R, "curve.png"), dpi=220); plt.close(fig)

# (2) 최종 정확도 막대
names = ["작은 ViT\n(처음부터)", "ResNet-20\n(처음부터)"]
vals = [vit["epochs"][-1]["test_acc"], res["epochs"][-1]["test_acc"]]
cols = [C_VIT, C_RES]
if os.path.exists(os.path.join(R, "vit_nopos.json")):
    names.insert(0, "작은 ViT\n(위치 임베딩 없음)"); vals.insert(0, load("vit_nopos.json")["epochs"][-1]["test_acc"]); cols.insert(0, "#F0B48A")
if pre:
    names.append("ViT-B/16\n(사전학습)"); vals.append(pre["test_acc"]); cols.append(C_PRE)
fig, ax = plt.subplots(figsize=(6.8, 4.4))
bars = ax.bar(names, vals, color=cols, width=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", va="bottom", fontsize=13, fontweight="bold")
ax.set_ylabel("CIFAR-10 테스트 정확도 (%)"); ax.set_ylim(0, 108); ax.set_yticks(range(0, 101, 20))
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(R, "bars.png"), dpi=220); plt.close(fig)
print(dict(zip([n.replace("\n", " ") for n in names], vals)))

# (3) CIFAR-10 예시 이미지
import torchvision
ds = torchvision.datasets.CIFAR10(os.path.join(os.path.expanduser("~"), ".cache", "vit_exp"), train=True, download=True)
fig, axes = plt.subplots(10, 7, figsize=(6.2, 6.6), gridspec_kw={"width_ratios": [1.6] + [1] * 6})
for c in range(10):
    idx = [i for i, t in enumerate(ds.targets[:2000]) if t == c][:6]
    axes[c, 0].text(0.95, 0.5, ds.classes[c], ha="right", va="center", fontsize=11)
    axes[c, 0].axis("off")
    for ax, i in zip(axes[c, 1:], idx):
        ax.imshow(ds.data[i], interpolation="nearest"); ax.axis("off")
fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.06, hspace=0.06)
fig.savefig(os.path.join(R, "cifar_samples.png"), dpi=220); plt.close(fig)


## 실험 1: 처음부터 학습 (ResNet-20 대 작은 ViT)

In [ ]:
!python exp1_scratch.py --model resnet --epochs 30

In [ ]:
!python exp1_scratch.py --model vit --epochs 30 --dim 192 --heads 3

위치 임베딩을 뺀 ViT (선택)

In [ ]:
!python exp1_scratch.py --model vit --no-pos --epochs 30 --dim 192 --heads 3

## 실험 2: 사전학습된 ViT-B/16 (linear probe)

In [ ]:
!python exp2_pretrained.py --train-size 50000 --test-size 10000

## 실험 3: 시각화

In [ ]:
!mkdir -p sample_images
!wget -q -O sample_images/dog.jpg https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg
!wget -q -O sample_images/astronaut.jpg https://raw.githubusercontent.com/pytorch/vision/main/gallery/assets/astronaut.jpg
!wget -q -O sample_images/dog2.jpg https://raw.githubusercontent.com/pytorch/vision/main/gallery/assets/dog2.jpg
!python exp3_visualize.py --images sample_images/dog.jpg sample_images/astronaut.jpg sample_images/dog2.jpg

## 결과 그래프

In [ ]:
!python plot_results.py
from IPython.display import Image, display
for f in ['bars.png', 'curve.png', 'pos_embed_similarity.png', 'attention_rollout.png']:
    display(Image('results/' + f, width=600))

## 결과 내려받기

내려받은 `results.zip`의 압축을 풀어 `실험/results` 폴더에 넣으면 됩니다.

In [ ]:
!zip -q -r results.zip results -x '*.pt'
from google.colab import files
files.download('results.zip')